In [ ]:
import scanpy as sc
import numpy as np
import os
from glob import glob

# -----------------------------
# Input & Output directories
# -----------------------------
input_dir = "dropout_h5ad"
output_dir = "imputed_mean"
os.makedirs(output_dir, exist_ok=True)

# -----------------------------
# Find all .h5ad files
# -----------------------------
files = glob(os.path.join(input_dir, "*.h5ad"))

# -----------------------------
# Mean Imputation for each file (zeros → mean)
# -----------------------------
for f in files:
    print(f"Processing {f} ...")
    # Load dropout dataset
    adata_missing = sc.read_h5ad(f)
    
    # Copy
    adata_mean = adata_missing.copy()
    
    # Convert sparse to dense (if needed)
    X = adata_mean.X.toarray() if not isinstance(adata_mean.X, np.ndarray) else adata_mean.X.copy()
    
    # Compute mean per gene (ignoring zeros)
    gene_means = np.true_divide(
        X.sum(axis=0),
        (X != 0).sum(axis=0),
        where=(X != 0).sum(axis=0) != 0  # avoid div/0
    )
    
    # Replace zeros with corresponding gene mean
    zero_inds = np.where(X == 0)
    X[zero_inds] = np.take(gene_means, zero_inds[1])
    
    # Assign back
    adata_mean.X = X
    
    # Save with matching name
    fname = os.path.basename(f).replace("adata_dropout", "adata_mean_imputed")
    outpath = os.path.join(output_dir, fname)
    adata_mean.write(outpath)
    
    print(f"Saved imputed file to: {outpath}")

print("All files processed with Mean Imputation (zeros → mean)")


In [ ]:
import scanpy as sc
import numpy as np
import os
from glob import glob
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor

# -----------------------------
# Input & Output directories
# -----------------------------
input_dir = "dropout_h5ad"
output_dir = "imputed_iterative"
os.makedirs(output_dir, exist_ok=True)

files = glob(os.path.join(input_dir, "*.h5ad"))

# Iterative Imputer with faster estimator
imputer = IterativeImputer(
    estimator=ExtraTreesRegressor(n_estimators=10, random_state=0, n_jobs=-1),
    max_iter=2,
    random_state=0
)

for f in files:
    print(f"\nProcessing {f} ...")

    adata_missing = sc.read_h5ad(f)
    adata_missing.obs_names_make_unique()
    adata_missing.var_names_make_unique()

    # Copy for imputation
    adata_imp = adata_missing.copy()

    # Step 1: Identify HVGs
    sc.pp.highly_variable_genes(adata_imp, n_top_genes=200, flavor="cell_ranger")
    hvg_mask = adata_imp.var["highly_variable"].values
    print(f"Selected {hvg_mask.sum()} HVGs out of {adata_imp.n_vars} genes")

    # Step 2: Convert to dense
    X = adata_imp.X.toarray() if not isinstance(adata_imp.X, np.ndarray) else adata_imp.X.copy()

    # Step 3: Replace zeros with NaN (treat as missing)
    X[X == 0] = np.nan

    # Step 4: Impute only HVGs
    X_hvg = X[:, hvg_mask]
    valid_genes = ~(np.all(np.isnan(X_hvg), axis=0))
    X_hvg_valid = X_hvg[:, valid_genes]

    print(f"Running imputation on HVGs matrix {X_hvg_valid.shape} ...")
    X_hvg_imp_valid = imputer.fit_transform(X_hvg_valid)

    # Step 5: Reconstruct HVG matrix
    X_hvg_imp = np.zeros_like(X_hvg)
    X_hvg_imp[:, valid_genes] = X_hvg_imp_valid
    # dropped HVGs stay zero

    # Step 6: Insert HVGs back into full matrix
    X_final = X.copy()
    X_final[:, hvg_mask] = X_hvg_imp

    # Step 7: Assign back
    adata_imp.X = X_final

    # Save
    fname = os.path.basename(f).replace("adata_dropout", "adata_iterative_imputed")
    outpath = os.path.join(output_dir, fname)
    adata_imp.write(outpath)

    print(f"✅ Saved imputed file to: {outpath}")

print("\nAll files processed with Iterative Imputation (HVGs only)")


In [ ]:
pip install --user scikit-misc

In [ ]:
import scanpy as sc
import numpy as np
import os
from glob import glob
from sklearn.impute import KNNImputer

# -----------------------------
# Input & Output directories
# -----------------------------
input_dir = "dropout_h5ad"
output_dir = "imputed_knn"
os.makedirs(output_dir, exist_ok=True)

files = glob(os.path.join(input_dir, "*.h5ad"))

# Define KNNImputer
imputer = KNNImputer(
    n_neighbors=10,      # number of neighbors to use
    weights="uniform",   # could also try "distance"
    metric="nan_euclidean"
)

for f in files:
    print(f"\nProcessing {f} ...")

    adata_missing = sc.read_h5ad(f)
    adata_missing.obs_names_make_unique()
    adata_missing.var_names_make_unique()

    # Copy for imputation
    adata_imp = adata_missing.copy()

    # Step 1: Identify HVGs
    sc.pp.highly_variable_genes(adata_imp, n_top_genes=200, flavor="cell_ranger")
    hvg_mask = adata_imp.var["highly_variable"].values
    print(f"Selected {hvg_mask.sum()} HVGs out of {adata_imp.n_vars} genes")

    # Step 2: Convert to dense
    X = adata_imp.X.toarray() if not isinstance(adata_imp.X, np.ndarray) else adata_imp.X.copy()

    # Step 3: Replace zeros with NaN (treat as missing)
    X[X == 0] = np.nan

    # Step 4: Impute only HVGs
    X_hvg = X[:, hvg_mask]
    valid_genes = ~(np.all(np.isnan(X_hvg), axis=0))  # drop all-NaN genes
    X_hvg_valid = X_hvg[:, valid_genes]

    print(f"Running KNNImputer on HVGs matrix {X_hvg_valid.shape} ...")
    X_hvg_imp_valid = imputer.fit_transform(X_hvg_valid)

    # Step 5: Reconstruct HVG matrix
    X_hvg_imp = np.zeros_like(X_hvg)
    X_hvg_imp[:, valid_genes] = X_hvg_imp_valid
    # dropped HVGs stay zero

    # Step 6: Insert HVGs back into full matrix
    X_final = X.copy()
    X_final[:, hvg_mask] = X_hvg_imp

    # Step 7: Assign back
    adata_imp.X = X_final

    # Save
    fname = os.path.basename(f).replace("adata_dropout", "adata_knn_imputed")
    outpath = os.path.join(output_dir, fname)
    adata_imp.write(outpath)

    print(f"✅ Saved imputed file to: {outpath}")

print("\nAll files processed with KNN Imputation (HVGs only)")


In [ ]:
import scanpy as sc
import numpy as np
import os
from glob import glob
from fancyimpute import SoftImpute

# -----------------------------
# Input & Output directories
# -----------------------------
input_dir = "dropout_h5ad"
output_dir = "imputed_softimpute"
os.makedirs(output_dir, exist_ok=True)

files = glob(os.path.join(input_dir, "*.h5ad"))

# Define SoftImpute
imputer = SoftImpute(
    max_rank=None,      # None means full rank, or set to e.g. 100
    init_fill_method="zero",  # starting point for missing values
    max_iters=100,
    verbose=True,
    convergence_threshold=1e-5
)

for f in files:
    print(f"\nProcessing {f} ...")

    adata_missing = sc.read_h5ad(f)
    adata_missing.obs_names_make_unique()
    adata_missing.var_names_make_unique()

    # Copy for imputation
    adata_imp = adata_missing.copy()

    # Step 1: Identify HVGs
    sc.pp.highly_variable_genes(adata_imp, n_top_genes=200, flavor="cell_ranger")
    hvg_mask = adata_imp.var["highly_variable"].values
    print(f"Selected {hvg_mask.sum()} HVGs out of {adata_imp.n_vars} genes")

    # Step 2: Convert to dense
    X = adata_imp.X.toarray() if not isinstance(adata_imp.X, np.ndarray) else adata_imp.X.copy()

    # Step 3: Replace zeros with NaN (treat as missing)
    X[X == 0] = np.nan

    # Step 4: Impute only HVGs
    X_hvg = X[:, hvg_mask]
    valid_genes = ~(np.all(np.isnan(X_hvg), axis=0))  # drop all-NaN genes
    X_hvg_valid = X_hvg[:, valid_genes]

    print(f"Running SoftImpute on HVGs matrix {X_hvg_valid.shape} ...")
    X_hvg_imp_valid = imputer.fit_transform(X_hvg_valid)

    # Step 5: Reconstruct HVG matrix
    X_hvg_imp = np.zeros_like(X_hvg)
    X_hvg_imp[:, valid_genes] = X_hvg_imp_valid
    # dropped HVGs stay zero

    # Step 6: Insert HVGs back into full matrix
    X_final = X.copy()
    X_final[:, hvg_mask] = X_hvg_imp

    # Step 7: Assign back
    adata_imp.X = X_final

    # Save
    fname = os.path.basename(f).replace("adata_dropout", "adata_softimpute_imputed")
    outpath = os.path.join(output_dir, fname)
    adata_imp.write(outpath)

    print(f"✅ Saved imputed file to: {outpath}")

print("\nAll files processed with SoftImpute (HVGs only)")


In [ ]:
pip install fancyimpute


In [ ]:
import scanpy as sc
import torch
import torch.nn as nn
import os
from glob import glob

# -----------------------------
# Simple GAN for Imputation
# -----------------------------
class Generator(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

class Discriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# -----------------------------
# Imputation function
# -----------------------------
def gan_impute(adata, epochs=100, lr=1e-4):
    # Convert sparse matrix → dense numpy
    if hasattr(adata.X, "toarray"):  
        X = adata.X.toarray()  
    else:
        X = adata.X  
    
    X = torch.tensor(X, dtype=torch.float32)

    input_dim = X.shape[1]
    G, D = Generator(input_dim), Discriminator(input_dim)

    g_opt = torch.optim.Adam(G.parameters(), lr=lr)
    d_opt = torch.optim.Adam(D.parameters(), lr=lr)
    loss_fn = nn.BCELoss()

    for epoch in range(epochs):
        # Train discriminator
        real = X
        noise = torch.randn_like(real)
        fake = G(noise)

        d_real = D(real)
        d_fake = D(fake.detach())
        d_loss = loss_fn(d_real, torch.ones_like(d_real)) + loss_fn(d_fake, torch.zeros_like(d_fake))
        d_opt.zero_grad()
        d_loss.backward()
        d_opt.step()

        # Train generator
        fake = G(noise)
        d_fake = D(fake)
        g_loss = loss_fn(d_fake, torch.ones_like(d_fake))
        g_opt.zero_grad()
        g_loss.backward()
        g_opt.step()

    # Use generator output as imputation
    imputed = G(torch.randn_like(X)).detach().numpy()
    adata.layers["imputed"] = imputed
    return adata


# -----------------------------
# Run GAN Imputation
# -----------------------------
input_dir = "dropout_h5ad"
output_dir = "imputed_gan"
os.makedirs(output_dir, exist_ok=True)

files = glob(os.path.join(input_dir, "*.h5ad"))

for f in files:
    print(f"Processing {f} with GAN...")
    adata = sc.read_h5ad(f)

    adata = gan_impute(adata, epochs=200)

    out = os.path.join(output_dir, os.path.basename(f).replace("dropout", "adata_gan_imputed"))
    adata.write_h5ad(out)
    print(f"✅ Saved {out}")


In [ ]:
import scanpy as sc
import scvi
import numpy as np
import os
from glob import glob

# -----------------------------
# Input & Output directories
# -----------------------------
input_dir = "dropout_h5ad"
output_dir = "imputed_scvae"
os.makedirs(output_dir, exist_ok=True)

files = glob(os.path.join(input_dir, "*.h5ad"))

for f in files:
    print(f"\nProcessing {f} with scVAE ...")

    adata = sc.read_h5ad(f)
    adata.obs_names_make_unique()
    adata.var_names_make_unique()

    # -----------------------------
    # Setup scVI
    # -----------------------------
    scvi.model.SCVI.setup_anndata(
        adata,
        layer=None,            # counts are in adata.X
        categorical_covariate_keys=None,
        continuous_covariate_keys=None
    )

    # -----------------------------
    # Train SCVI model
    # -----------------------------
    model = scvi.model.SCVI(
        adata,
        n_latent=30,           # latent dimensions
        gene_likelihood="zinb" # scRNA-seq usually ZINB
    )
    model.train(
        max_epochs=400,
        early_stopping=True,
        early_stopping_patience=45
    )

    # -----------------------------
    # Get latent representation
    # -----------------------------
    Z = model.get_latent_representation()
    adata.obsm["X_scVI"] = Z

    # -----------------------------
    # Denoised (imputed) expression
    # -----------------------------
    denoised = model.get_normalized_expression(library_size=1e4)
    adata.layers["scVI_imputed"] = denoised.values

    # -----------------------------
    # Save
    # -----------------------------
    fname = os.path.basename(f).replace("adata_dropout", "adata_scVI_imputed")
    outpath = os.path.join(output_dir, fname)
    adata.write(outpath)

    print(f"✅ Saved scVAE-imputed file to: {outpath}")

print("\nAll files processed with scVAE (scVI) imputation")


In [ ]:
import packaging
print(packaging.__version__)


In [ ]:
pip install packaging==25.0


In [ ]:
pip install --force-reinstall --no-deps packaging==23.2
